# X6Y3 manual qcal tune-up — Q0 and Q1

Use the **X6Y3 QubiC (local)** kernel. Run the Preamble and Readout Calibration first, then the desired calibration cells individually. Measurement cells run real experiments when you execute them; nothing starts merely by opening this notebook.

This copy includes the working host/RPC, qcal settings and fresh-classifier fixes, restricted to Q0/Q1 on the two-core aa01c78f image. RPC: `127.0.0.1:9095` → Huracan `192.168.1.236:9095`. Active/unconditional reset and heralding are disabled; passive relaxation is retained. GE/EF readout, frequency, amplitude, phase, T1 and T2 sections are available. Start with GE; EF sections train a three-state classifier.

Calibration `.run()` methods may save accepted fits into this notebook's **private configuration** at `/local/data/projects/LeeQ-lbnl-qubic/notebooks/RealSystem/configs_X6Y3_qcal_Q0Q1`. Original supplied configuration is preserved. Data go to `data_X6Y3_qcal_Q0Q1` beside this notebook. Reopening uses the existing private calibration; it does not recopy the original settings. Set `QUBITS` to `[0]`, `[1]`, or `[0, 1]` in the Preamble before selecting a section.

The original CZ/coupler sections are retained in the backup, because this selected two-core configuration has no coupler-drive mapping. Original uploaded notebook backup: `Calibration_X6Y3_LBNL_original.20260929T064019Z.ipynb.bak`.

Notebook cells were not executed during replacement. Offline syntax and Q0/Q1 compilation are checked separately; calibration fit quality must be assessed from your measurements.


# Preamble

In [ ]:
from pathlib import Path
import sys

HARNESS_ROOT = Path('/local/data/projects/qubic_validation')
SOFTWARE_ROOT = HARNESS_ROOT / 'artifacts/x6y3/signature-software'
sys.path[:0] = [str(SOFTWARE_ROOT), str(HARNESS_ROOT / 'X6Y3')]

import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import scipy
import qcal as qc
from qcal.units import *
from qcal.utils import load_from_pickle
from qcal.backend.qubic.qpu import QubicQPU
from qcal.backend.qubic.utils import calculate_n_reads, qubic_sequence
from qcal.benchmarking.rb import SRB, CRB
from qcal.benchmarking.pb import XRB
from qcal.benchmarking.cb import CB
from qcal.benchmarking.pnr import KNR
from qcal.benchmarking.readout import ReadoutFidelity
from qcal.calibration.readout import ReadoutCalibration, Separation, Fidelity
from qcal.calibration.single_qubit import Amplitude, Frequency, Phase
from qcal.characterization.spectroscopy.resonator import Resonator, Punchout
from qcal.characterization.tomography.truth_table import TruthTable
from qcal.calibration.cz import RelativePhase
from qcal.characterization.coherence import T1, T2, ParityOscillations
from qcal.characterization.phase_estimation.rpe import RPE
from qcal.calibration.leakage import Leakage

from qcal.fitting.fit import FitLinear
from qcal.optimization.optimize import Optimize
from qcal.optimization.optimizers import Adam, CMA, LQR

import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

logging.basicConfig(level=logging.INFO)

%load_ext autoreload
%autoreload 2
from classifier_utils import fresh_readout_calibration


In [ ]:
import qcal.settings as settings

NOTEBOOK_DIR = Path('/local/data/projects/LeeQ-lbnl-qubic/notebooks/RealSystem')
CONFIG_DIR = NOTEBOOK_DIR / 'configs_X6Y3_qcal_Q0Q1'
DATA_DIR = NOTEBOOK_DIR / 'data_X6Y3_qcal_Q0Q1'
DATA_DIR.mkdir(exist_ok=True)
settings.Settings.config_path = str(CONFIG_DIR) + '/'
settings.Settings.data_save_path = str(DATA_DIR) + '/'
settings.Settings.data_path = settings.Settings.data_save_path
settings.Settings.save_data = True

QUBITS = [0, 1]  # Or [0] / [1] for one qubit at a time.
assert QUBITS and set(QUBITS) <= {0, 1}
ip_address, port = '127.0.0.1', 9095
sd_param = None
classifier_path = CONFIG_DIR / 'ClassificationManager.pkl'
classifier = load_from_pickle(str(classifier_path)) if classifier_path.exists() else None
cfg = qc.Config()
assert set(cfg.qubits) == {0, 1}, cfg.qubits
print('Selected qubits:', QUBITS, '| Configuration:', CONFIG_DIR)
print('RPC:', ip_address, port, '| Data:', DATA_DIR)
cfg.draw_qpu()


# Readout Calibration

In [ ]:

cfg.load()

cal = fresh_readout_calibration(
    QubicQPU,
    cfg,
    qubits=QUBITS,
    gate='X90',
    n_circs_per_seq=3,
    n_shots=5000,
    n_levels=2,
    save_raw_data=True,
    ip_address=ip_address,
    port=port,
  )
cal.run()

classifier = cal.classifier


## Separation

In [ ]:
cfg.load()

qubits = list(QUBITS)
params = {q: f'readout/{q}/freq' for q in qubits}
param_sweep = { # +/- delta MHz sweep around the current frequency
    q: np.linspace(-2.5, 2.5, 31) * MHz
    + cfg[f'readout/{q}/freq'] for q in qubits
}

    

cal = Separation(
    QubicQPU,
    cfg,
    qubits=qubits,
    params=params,
    param_sweep=param_sweep,
    n_circs_per_seq=3,
    n_shots=3000,
    n_levels=2,
    save_raw_data=True,
    ip_address=ip_address,
    port=port,
    gate='X90',
)
cal.run()


In [ ]:
cfg.load()

_readout_training = fresh_readout_calibration(
    QubicQPU, cfg, qubits=QUBITS, gate='X90',
    n_circs_per_seq=3, n_shots=5000, n_levels=2,
    save_raw_data=True, ip_address=ip_address, port=port,
)
_readout_training.run(plot=False)
classifier = _readout_training.classifier


qubits = list(QUBITS)



cfg['readout/esp/enable'] = False
params = {q: f'readout/{q}/amp' for q in qubits}
param_sweep = {
    q: np.linspace(-0.005, 0.005, 31)
    + cfg[f'readout/{q}/amp'] for q in qubits
}



cal = Fidelity(
    QubicQPU,
    cfg,
    qubits=qubits,
    params=params,
    param_sweep=param_sweep,
    classifier=classifier,
    n_circs_per_seq=3,
    n_shots=3000,
    n_levels=2,
    save_raw_data=False,
    ip_address=ip_address,
    port=port,
    gate='X90',
  )
cal.run()


# Readout Fidelity

In [ ]:
cfg.load()

_readout_training = fresh_readout_calibration(
    QubicQPU, cfg, qubits=QUBITS, gate='X90',
    n_circs_per_seq=3, n_shots=5000, n_levels=2,
    save_raw_data=True, ip_address=ip_address, port=port,
)
_readout_training.run(plot=False)
classifier = _readout_training.classifier


qubits = list(QUBITS)
ro = ReadoutFidelity(QubicQPU, cfg, qubits, classifier=classifier, raster_circuits=False,
                     n_circs_per_seq=3, n_shots=5000, n_levels=2, ip_address=ip_address, port=port, gate='X90')
ro.run()
ro.confusion_matrix


# Single Qubit

## GE

### Freq

In [ ]:
from qcal.calibration.single_qubit import Frequency


In [ ]:
cfg.load()

t_max = 1*us
detunings = np.array([-5, -2.5, 2.5, 5])*MHz



freq = Frequency(
    QubicQPU,
    cfg,
    qubits=QUBITS,
    t_max=t_max,
    detunings=detunings,
    classifier=classifier,
    n_shots=512,
    n_circs_per_seq=200,
    reload_freq=False,
    reload_env=False,
    zero_between_reload=False,
    ip_address=ip_address,
    port=port
)
freq.run()


### Amplitude

In [ ]:
qubits = list(QUBITS)


In [ ]:
cfg.load()
qubits = list(QUBITS)

amps = np.linspace(0, 1.0, 31)
cal = Amplitude(
    QubicQPU,
    cfg,
    qubits,
    gate='X90',
    subspace='GE',
    amplitudes=amps,
    n_gates=1,
    n_circs_per_seq=amps.size,
    classifier=classifier,
    ip_address=ip_address,
    port=port
)
cal.run()


In [ ]:
cfg.load()
qubits = list(QUBITS)

amps = np.linspace(0.7, 1.3, 31)
cal = Amplitude(
    QubicQPU,
    cfg,
    qubits,
    gate='X90',
    subspace='GE',
    amplitudes=amps,
    n_gates=4,
    n_shots=512,
    n_circs_per_seq=amps.size,
    classifier=classifier,
    relative_amp=True,
    ip_address=ip_address,
    port=port
)
cal.run()


### Phase

In [ ]:
qubits = list(QUBITS)


In [ ]:
cfg.load()

phases = np.linspace(-0.25, 0.25, 21)

for i in QUBITS:
    qubits = [i]
    cal = Phase(
        QubicQPU,
        cfg,
        classifier=classifier,
        qubits=qubits,
        phases=phases,
        subspace='GE',
        relative_phase=True,
        n_circs_per_seq=50,
        ip_address=ip_address,
        port=port
    )
    cal.run()


In [ ]:
cfg.load()
qubits = list(QUBITS)

phases = np.linspace(-0.3, 0.3, 31)

cal = Phase(
    QubicQPU,
    cfg,
    classifier=classifier,
    qubits=qubits,
    phases=phases,
    subspace='GE',
    n_circs_per_seq=phases.size,
    ip_address=ip_address,
    port=port
)
cal.run()


## EF

### Frequency

In [ ]:
cfg.load()

_readout_training = fresh_readout_calibration(
    QubicQPU, cfg, qubits=QUBITS, gate='X90',
    n_circs_per_seq=3, n_shots=5000, n_levels=3,
    save_raw_data=True, ip_address=ip_address, port=port,
)
_readout_training.run(plot=False)
classifier = _readout_training.classifier


t_max = 0.5*us
detunings = np.array([-5, -2.5, 2.5, 5])*MHz

cfg['readout/esp/enable'] = False
freq = Frequency(
    QubicQPU,
    cfg,
    qubits=QUBITS,
    t_max=t_max,
    detunings=detunings,
    subspace='EF',
    classifier=classifier,
    n_shots=512,
    n_circs_per_seq=200,
    reload_freq=False,
    reload_env=False,
    zero_between_reload=False,
    ip_address=ip_address,
    port=port
)
freq.run()


### AMP

In [ ]:
qubits = list(QUBITS)


In [ ]:
cfg.load()
qubits = list(QUBITS)

if classifier[classifier._qubits[0]].n_components < 3:
    _readout_training = fresh_readout_calibration(
        QubicQPU, cfg, qubits=QUBITS, gate='X90',
        n_circs_per_seq=3, n_shots=5000, n_levels=3,
        save_raw_data=True, ip_address=ip_address, port=port,
    )
    _readout_training.run(plot=False)
    classifier = _readout_training.classifier


amps = {q: np.linspace(0, 0.25, 21) for q in QUBITS}

cal = Amplitude(
      QubicQPU,
      cfg,
      qubits,
      gate='X90',
      subspace='EF',
      amplitudes=amps,
      n_circs_per_seq=21,
      classifier=classifier,
      ip_address=ip_address,
      port=port
)
cal.run()


In [ ]:
cfg.load()
qubits = list(QUBITS)

amps = np.linspace(0.7, 1.3, 21)
cal = Amplitude(
      QubicQPU,
      cfg,
      qubits,
      gate='X90',
      subspace='EF',
      amplitudes=amps,
      n_gates=4,
      n_shots=512,
      n_circs_per_seq=amps.size,
      classifier=classifier,
      relative_amp=True,
      ip_address=ip_address,
      port=port
)
cal.run()


# Coherences

## T1

### GE

In [ ]:
cfg.load()

cfg['readout/esp/enable'] = False
qubits = list(QUBITS)
n_elements = 50
char = T1(
    QubicQPU,
    cfg,
    qubits,
    t_max=350*us,
    classifier=classifier,
    n_elements=n_elements,
    n_circs_per_seq=n_elements,
    n_shots=2000,
    reload_freq=False,
    reload_env=False,
    zero_between_reload=False,
    ip_address=ip_address,
    port=port
  )
char.run()


### EF

In [ ]:
cfg.load()

qubits = list(QUBITS)
n_elements = 50
cfg['readout/esp/enable'] = False
char0 = T1(
  QubicQPU,
  cfg,
  qubits,
  t_max=200*us,
  subspace='EF',
  classifier=classifier,
  n_elements=n_elements,
  n_circs_per_seq=n_elements,
  n_shots=2000,
  reload_freq=False,
  reload_env=False,
  zero_between_reload=False,
  sd_param=sd_param,
  ip_address=ip_address,
  port=port
)
char0.run()


## T2*

### GE

In [ ]:
cfg.load()

qubits = list(QUBITS)
n_elements = 50
char = T2(
    QubicQPU,
    cfg,
    qubits,
    t_max=100*us,
    echo=False,
    detuning=40*kHz,
    classifier=classifier,
    n_elements=n_elements,
    n_circs_per_seq=n_elements,
    reload_freq=False,
    reload_env=False,
    zero_between_reload=False,
    ip_address=ip_address,
    port=port
  )
char.run()


### EF

In [ ]:
cfg.load()

qubits = list(QUBITS)
n_elements = 50
char = T2(
    QubicQPU,
    cfg,
    qubits,
    t_max=250*us,
    echo=False,
    detuning=25*kHz,
    subspace='EF',
    classifier=classifier,
    n_elements=n_elements,
    n_circs_per_seq=n_elements,
    reload_freq=False,
    reload_env=False,
    zero_between_reload=False,
    sd_param=sd_param,
    ip_address=ip_address,
    port=port
)
char.run()
